# Network Diagnostics in pandapipes

A pipeflow calculation may fail because of incorrect network data, unrealistic
component parameters, missing connections or unsuitable calculation settings.

Finding the cause manually can be difficult, especially in larger networks.
The pandapipes diagnostic framework performs a collection of checks and
provides information about possible causes of non-convergence.

## Diagnostic Checks

The diagnostic framework checks different areas of a pandapipes network,
including:

- input data and network structure
- plausibility of component parameters
- possible causes of non-convergence
- calculation modes and solver settings

Some checks identify definite data errors, while others test whether changing
a parameter or network configuration restores convergence.

## How the Diagnostic Framework Works

Some checks inspect the network tables directly and report invalid values,
missing elements or incorrect references.

If the original pipeflow does not converge, other checks create temporary
copies of the network. They change one parameter or configuration and run the
pipeflow again. This helps determine whether the tested change restores
convergence.

The original network is not modified or automatically repaired. The diagnostic
results indicate which network data or calculation settings should be reviewed
by the user.

## Running the Diagnostic

All default diagnostic checks can be executed with one function call:

```python
results = pp.diagnostic(net)
```

The function prints a readable diagnostic report and returns the results of
the applicable checks as a dictionary.

Checks that return `None` are omitted. Depending on the check, the stored
result may be a Boolean value, a list or another dictionary containing more
detailed information.

Several diagnostic parameters can be adjusted directly when calling the
function. Parameters that are not specified retain their default values.

For example:

```python
results = pp.diagnostic(
    net,
    pipe_length_percentage=20.0,
    minimum_pipe_length_km=2.0,
    standard_pipe_length_km=0.05,
    iteration_limit=300,
)
```

## Practical Example

### 1. Creating a Healthy Gas Network

First, a small gas network with six junctions, five pipes, one valve, one
external grid, one sink and one source is created.

In [7]:
import pandapipes as pp

In [8]:
net = pp.create_empty_network(fluid="lgas")

j1 = pp.create_junction(net, 1.05, 293.15, height_m=0)
j2 = pp.create_junction(net, 1.05, 293.15, height_m=0)
j3 = pp.create_junction(net, 1.05, 293.15, height_m=0)
j4 = pp.create_junction(net, 1.05, 293.15, height_m=0)
j5 = pp.create_junction(net, 1.05, 293.15, height_m=0)
j6 = pp.create_junction(net, 1.05, 293.15, height_m=0)

pp.create_ext_grid(net, junction=j1, p_bar=1.1, t_k=293.15, )

pp.create_pipe_from_parameters(net, j1, j2, length_km=10, inner_diameter_mm=300, )
pp.create_pipe_from_parameters(net, j2, j3, length_km=2, inner_diameter_mm=300, )
pp.create_pipe_from_parameters(net, j2, j4, length_km=2.5, inner_diameter_mm=300, )
pp.create_pipe_from_parameters(net, j3, j5, length_km=1, inner_diameter_mm=300, )
pp.create_pipe_from_parameters(net, j4, j6, length_km=1, inner_diameter_mm=300, )

pp.create_valve(net, junction=j5, element=j6, et="ju", inner_diameter_mm=50, opened=True, )
pp.create_sink(net, junction=j4, mdot_kg_per_s=0.545, )

pp.create_source(net, junction=j3, mdot_kg_per_s=0.234, );

First, the pipeflow and the diagnostic are executed on the original network.

In [9]:
pp.pipeflow(net)

print("Converged:", net.converged)

healthy_results = pp.diagnostic(net)
healthy_results

Converged: True




---------------- PANDAPIPES DIAGNOSTIC TOOL ----------------



--------------- END OF PANDAPIPES DIAGNOSTIC ---------------



{}

The empty result dictionary indicates that no relevant problem was found in
the healthy network.

### 2. Invalid Input Data

The length of one pipe is changed to an invalid negative value.

In [10]:
net_invalid = net.deepcopy()
net_invalid.pipe.loc[net_invalid.pipe.index[0], "length_km",] = -1.0

invalid_results = pp.diagnostic(net_invalid)
invalid_results



---------------- PANDAPIPES DIAGNOSTIC TOOL ----------------

invalid_values:
pipe:
pipe 0: 'length_km' = -1.0 (restriction: >0)


--------------- END OF PANDAPIPES DIAGNOSTIC ---------------



{'invalid_values': {'pipe': [(0, 'length_km', -1.0, '>0')]}}

The report identifies the affected pipe, the invalid parameter value and the
required restriction.

### 3. Non-Convergence Caused by External-Grid Pressure

The external-grid pressure is set to an extremely high value, causing the
pipeflow calculation to fail.

In [5]:
from pandapipes import PipeflowNotConverged

net_pressure = net.deepcopy()
net_pressure.ext_grid.p_bar = 100000

try:
    pp.pipeflow(net_pressure)

except PipeflowNotConverged:
    print("The pipeflow did not converge.")

The pipeflow did not converge.


The diagnostic framework tests whether changing the external-grid pressure
restores convergence.

In [6]:
pressure_results = pp.diagnostic(net_pressure)
pressure_results



---------------- PANDAPIPES DIAGNOSTIC TOOL ----------------

Checking ext_grid pressure...

pipeflow converges if ext_grid pressures are scaled by a factor of 1e-05.

------------------------------------------------------------

Checking pipe lengths...

Pipeflow still does not converge if only the longest 10.0% of all pipes with lengths above 1.0 km are set to 0.01 km.
It converges if all pipe lengths are set to 0.01 km.

------------------------------------------------------------

Checking iteration limit...

The pipeflow converges if the maximum number of iterations is increased to 200.

------------------------------------------------------------

Checking sink/source scaling...

Pipeflow still does not converge if sinks and sources are scaled by a factor of 1e-05.

------------------------------------------------------------

Checking valve configuration...

Pipeflow still does not converge if all valves are opened.
If all valves were closed, the pipeflow would converge.

----

{'ext_grid_pressure': np.True_,
 'pipe_length': {'long_pipes': False, 'all_pipes': np.True_},
 'iteration_check': True,
 'sink_source_scaling': {'sink': False, 'source': False, 'both': False},
 'valve_configuration': {'all_open': False, 'all_closed': np.True_},
 'calculation_mode': {'hydraulics': False,
  'heat': False,
  'sequential': False,
  'bidirectional': False},
 'friction_model': {'nikuradse': False,
  'colebrook': False,
  'swamee-jain': False},
 'alpha_sweep': False}

Several diagnostic modifications restore convergence in this example. Since
the external-grid pressure was deliberately changed to an unrealistic value,
the `ext_grid_pressure` result is the relevant result here.

Other successful checks show that alternative numerical modifications can
also restore convergence. They do not prove that the corresponding network
parameters are incorrect.

## Conclusion

The diagnostic framework can detect invalid network data and investigate
possible causes of non-convergence. It does not automatically repair the
network. The reported results must be reviewed by the user, and all diagnostic
modifications are applied only to temporary copies of the original network.